# Round 2: PCA on the 1000G superpopulation, OADP projection, robust gate around the target population(s)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
def mahalanobis(Z, mu, C):
    d = Z[:, :len(mu)] - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", d, np.linalg.inv(C), d))

def gate_ellipse(ax, mu, C, i, j, r, **kw):
    """Outline of the ellipsoid on PCs i, j."""
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(C[np.ix_([i, j], [i, j])])
    ax.add_patch(Ellipse((mu[i], mu[j]), 2 * r * np.sqrt(vals[-1]), 2 * r * np.sqrt(vals[0]),
                         angle=np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1])), fill=False, **kw))

def fit_robust_gate(A, q, seed=0):
    """Reweighted MCD on the target pops; radius = sqrt(chi2_K quantile q). MCD, not OGK:
    OGK's scatter isn't consistency-corrected, so chi-square radii on it are off."""
    from scipy.stats import chi2
    from sklearn.covariance import MinCovDet
    mcd = MinCovDet(random_state=seed).fit(A)
    mu, C = mcd.location_, mcd.covariance_
    return {"mu": mu, "C": C, "radius": float(np.sqrt(chi2.ppf(q, A.shape[1]))),
            "ref_d": mahalanobis(A, mu, C), "anchor_outliers": int((~mcd.support_).sum())}

def peak_enrichment(loadings, pcs, window_mb=10):
    """Max over windows of (share of squared loading / share of variants); ~1 = spread out."""
    w = loadings["chr"].astype(str) + ":" + (loadings["pos"] // (window_mb * 1e6)).astype(int).astype(str)
    share_var = w.value_counts(normalize=True)
    out = {}
    for pc in pcs:
        sq = loadings[pc] ** 2
        share = sq.groupby(w).sum() / sq.sum()
        out[pc] = float((share / share_var.reindex(share.index)).max())
    return pd.Series(out)

def hexbin_safe(ax, x, y, **kw):
    """Participant density; cells with fewer than 21 people are not drawn."""
    return ax.hexbin(x, y, **{"gridsize": 120, "bins": "log", "cmap": "Greys", **kw, "mincnt": 21})

## PCA on the reference, OADP projection (bigsnpr)

In [ ]:
PCA_OUT = single_target(cfg, "round2_pca")
PCA_DIR = os.path.dirname(PCA_OUT)
require_ready(cfg, "round2_pca")
rep_pca = Report(cfg, "round2_pca")
RUN = needs_run(PCA_OUT, FORCE)
setenv(RUN=int(RUN), REF_BFILE=cfg.input("ref_bfile"), REF_LABELS=cfg.input("ref_labels"),
       SUPERPOP=cfg.param("superpop"), NEW_BED=single_target(cfg, "hm3_panel"),
       K=cfg.param("r2_n_pcs_fit"), LD_REGIONS=cfg.input("ld_regions"), OUT_DIR=PCA_DIR, THREADS=THREADS)

In [ ]:
%%script Rscript -
suppressMessages({ library(bigsnpr); library(data.table) })
if (Sys.getenv("RUN") != "1") { cat("exists, keeping it (FORCE = True to remake)\n"); quit(save = "no") }
env <- function(k) { v <- Sys.getenv(k); if (!nzchar(v)) stop("missing ", k); v }
ncores <- as.integer(env("THREADS"))

ld_mask <- function(map, path) {   # listed long-range LD regions
  ld <- fread(path, header = FALSE)[, V1 := sub("^chr", "", V1)]
  m <- rep(FALSE, nrow(map))
  for (r in seq_len(nrow(ld)))
    m <- m | (as.character(map$chromosome) == ld$V1[r] & map$physical.pos >= ld$V2[r] & map$physical.pos <= ld$V3[r])
  m
}

out_dir <- env("OUT_DIR"); dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
K <- as.integer(env("K"))
obj.ref <- bed(paste0(env("REF_BFILE"), ".bed"))
obj.new <- bed(env("NEW_BED"))

# training set: founders of the superpopulation only
lab <- fread(env("REF_LABELS"), colClasses = "character")[match(obj.ref$fam$sample.ID, SampleID)]
stopifnot(!anyNA(lab$SampleID))
ind.train <- which(lab$Superpopulation == env("SUPERPOP") &
                   lab$FatherID == "0" & lab$MotherID == "0")
cat(length(ind.train), "training founders\n")

# autoSVD on the reference (clump r2 0.2, iterative long-range-LD removal), then OADP
in_ld <- ld_mask(obj.ref$map, env("LD_REGIONS"))
proj <- bed_projectPCA(obj.ref, obj.new, k = K, ind.row.ref = ind.train, ind.col.ref = which(!in_ld),
                       strand_flip = TRUE, join_by_pos = TRUE, ncores = ncores)
svd <- proj$obj.svd.ref
used <- attr(svd, "subset")
lrldr <- attr(svd, "lrldr")
pcs <- paste0("PC", seq_len(K))
ref_scores <- predict(svd); colnames(ref_scores) <- pcs
oadp <- proj$OADP_proj; simple <- proj$simple_proj
colnames(oadp) <- colnames(simple) <- pcs

map <- obj.ref$map
fwrite(data.table(id = obj.new$fam$sample.ID, simple), file.path(out_dir, "simple_projection.tsv"), sep = "\t")
fwrite(data.table(chr = map$chromosome[used], pos = map$physical.pos[used], svd$v), file.path(out_dir, "loadings.tsv"), sep = "\t")
fwrite(if (is.null(lrldr)) data.table(Chr = integer(), Start = integer(), Stop = integer()) else as.data.table(lrldr),
       file.path(out_dir, "lrldr.tsv"), sep = "\t")
jsonlite::write_json(list(n_train = length(ind.train), n_variants_used = length(used),
                          n_listed_ld_excluded = sum(in_ld)), file.path(out_dir, "round2_meta.json"), auto_unbox = TRUE)
fwrite(rbind(data.table(id = lab$SampleID[ind.train], set = "reference", pop = lab$Population[ind.train], ref_scores),
             data.table(id = obj.new$fam$sample.ID, set = "cohort", pop = NA_character_, oadp)),
       file.path(out_dir, "scores.tsv"), sep = "\t")

In [ ]:
if RUN:
    mark_done("round2_pca", PCA_OUT, tools={"bigsnpr": "bed_projectPCA"})

meta = json.load(open(f"{PCA_DIR}/round2_meta.json"))
scores = pd.read_csv(PCA_OUT, sep="\t", dtype={"id": str})
simple = pd.read_csv(f"{PCA_DIR}/simple_projection.tsv", sep="\t", dtype={"id": str})
load = pd.read_csv(f"{PCA_DIR}/loadings.tsv", sep="\t")
lrldr = pd.read_csv(f"{PCA_DIR}/lrldr.tsv", sep="\t")
PCS = [c for c in scores if c.startswith("PC")]
load = load.rename(columns={f"V{k}": pc for k, pc in enumerate(PCS, start=1)})
ref_s, coh = scores[scores["set"] == "reference"], scores[scores["set"] == "cohort"]

rep_pca.note(superpop=cfg.param("superpop"), n_train=meta["n_train"],
             n_variants_used=meta["n_variants_used"], n_listed_ld_excluded=meta["n_listed_ld_excluded"])
rep_pca.check("fit on reference founders only (no relatives in the fit)", True, f"{meta['n_train']} founders")
rep_pca.check("long-range LD detected and removed (bed_autoSVD)", True,
              "; ".join(f"chr{r.Chr}:{r.Start/1e6:.0f}-{r.Stop/1e6:.0f} Mb" for r in lrldr.itertuples()) or "none found")

## Pick K: PCs with population structure and no loading peak

In [ ]:
r2_pop = {pc: ref_s.groupby("pop")[pc].transform("mean").var() / ref_s[pc].var() for pc in PCS}
diag = pd.DataFrame({"pc": PCS, "r2_population": [round(r2_pop[p], 3) for p in PCS],
                     "loading_peak": peak_enrichment(load, PCS).round(1).values,
                     "oadp_over_simple_sd": (coh[PCS].std().values / simple[PCS].std().values).round(2)})
rep_pca.table(diag, "pc_diagnostics")
diag

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(12, 9.6), sharex=True)
for ax, pc in zip(axes, PCS):
    ax.scatter(np.arange(len(load)), load[pc].abs(), s=0.5, c=load["chr"] % 2, cmap="coolwarm", rasterized=True)
    ax.set_ylabel(pc)
axes[-1].set_xlabel("variant (genome order)")
rep_pca.fig(fig, "loadings"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3), (4, 5)]):
    for p, d in ref_s.groupby("pop"):
        ax.scatter(d[PCS[i]], d[PCS[j]], s=8, label=p)
    ax.set(xlabel=PCS[i], ylabel=PCS[j])
axes[0].legend(fontsize=8)
rep_pca.fig(fig, "reference_scores", "1000G training samples"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, pc in zip(axes, PCS[:4]):
    ax.hist2d(simple[pc], coh[pc], bins=80, cmin=21, cmap="Blues")
    lim = np.abs(coh[pc]).quantile(0.995)
    ax.plot([-lim, lim], [-lim, lim], "k:", lw=0.8)
    ax.set(xlabel=f"{pc} simple", ylabel=f"{pc} OADP")
rep_pca.fig(fig, "simple_vs_oadp", "simple projection shrinks toward 0"); plt.show()
rep_pca.save()

## Gate (set `r2_k` and `r2_q` in the run config from the table above)

In [ ]:
STAGE = "ancestry_round2"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
K, Q, TARGET = cfg.param("r2_k"), cfg.param("r2_q"), cfg.param("target_pops")
missing = sorted(set(TARGET) - set(ref_s["pop"]))
rep.check("target populations are in the superpopulation", not missing, ", ".join(missing))
PCK = PCS[:K]
A = ref_s[ref_s["pop"].isin(TARGET)][PCK].to_numpy()
g = fit_robust_gate(A, Q)
d = mahalanobis(coh[PCK].to_numpy(), g["mu"], g["C"])
keep = d <= g["radius"]
d_simple = mahalanobis(simple[PCK].to_numpy(), g["mu"], g["C"])

rep.note(k=K, q=Q, target="+".join(TARGET), n_target=len(A), target_outliers=g["anchor_outliers"],
         radius=round(g["radius"], 3))
rep.count("n_round1", len(coh))
print("kept", rep.count("n_kept", keep.sum()))
rep.count("n_kept_if_simple_projection", (d_simple <= g["radius"]).sum())
rep.check("target population inside the gate", (g["ref_d"] <= g["radius"]).mean() >= 0.9,
          f"{(g['ref_d'] <= g['radius']).mean():.1%}")
rep.check("every PC used carries population structure", all(diag["r2_population"][:K] > 0.02))
rep.check("no PC used is driven by one region", all(diag["loading_peak"][:K] < 10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (i, j) in zip(axes, [(0, 1), (1, 2)]):
    if j >= K:
        ax.axis("off"); continue
    hexbin_safe(ax, coh[PCS[i]], coh[PCS[j]])
    for p, dd in ref_s.groupby("pop"):
        ax.scatter(dd[PCS[i]], dd[PCS[j]], s=8, marker="x", label=p)
    gate_ellipse(ax, g["mu"], g["C"], i, j, g["radius"], edgecolor="tab:red", lw=1.5)
    ax.set(xlabel=PCS[i], ylabel=PCS[j])
axes[0].legend(fontsize=8)
rep.fig(fig, "gate_pcs"); plt.show()

from scipy.stats import chi2
qs = np.array([0.9, 0.99, 0.999, 0.9999, 0.99999])
kept_q = [int((d <= np.sqrt(chi2.ppf(q, K))).sum()) for q in qs]
rep.table(pd.DataFrame({"q": qs, "kept": kept_q}), "kept_vs_q", counts=["kept"])

## Write

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT) as meta:
        get_store(OUT).write_text(OUT, "\n".join(coh.loc[keep, "id"]) + "\n")
        meta.update(n_kept=int(keep.sum()), k=K, q=Q)
rep.save()